In [2]:
# Libraries and Data Import
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

# Load Data
data = pd.read_csv('Processed_Wide_Lag_Ready.csv')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Data loaded. Shape: {data.shape}. Device: {device}")

Data loaded. Shape: (471, 354). Device: cuda


In [3]:
# --- Champion Model Configuration ---
config = {
    'lookback': 13,      # Lookback Window
    'horizon': 13,       # Forecast Window
    'hidden': 128,       # Hidden layer size
    'lr': 0.001,
    'dropout': 0.0,      # Zero dropout to reduce underfitting
    'epochs': 150,
    'patience': 20
}

total_weeks = len(data)
split_idx = int(total_weeks * 0.90)
print(f"Training on {split_idx} weeks, Validating on {total_weeks - split_idx} weeks.")

Training on 423 weeks, Validating on 48 weeks.


In [4]:
# --- Data Preparation ---
# Comment out this row to forecast on all data
data = data[data['Week'] < '2025-10-01']

dates = pd.to_datetime(data['Week'])
df_numeric = data.drop(columns=['Week'])
commodity_names = df_numeric.columns[:-1] # All except Total_Volume

full_data_matrix = df_numeric.values.astype(np.float32)

# Log transform the last column (Total_Volume)
full_data_matrix[:, -1] = np.log1p(full_data_matrix[:, -1])

scaler = StandardScaler()

# Scale only Train Data
scaler.fit(full_data_matrix[:split_idx])

full_data_matrix_scaled = full_data_matrix.copy()
full_data_matrix_scaled = scaler.transform(full_data_matrix)


print("Data scaled and Total_Volume log-transformed.")

Data scaled and Total_Volume log-transformed.


In [5]:
# --- MODEL ARCHITECTURE ---
class FreightLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, 128)
        self.lstm = nn.LSTM(128, hidden_dim, num_layers=1, batch_first=True)
        self.linear2 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        x_enc = self.relu(self.linear1(x.view(-1, x.size(-1))))
        x_enc = x_enc.view(batch_size, seq_len, -1)
        lstm_out, _ = self.lstm(x_enc)
        # Only take the last output of the sequence
        out = self.dropout(self.linear2(lstm_out[:, -1, :]))
        return out

class FreightDataset(Dataset):
    def __init__(self, data, lookback, horizon):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.lookback = lookback
        self.horizon = horizon

    def __len__(self):
        return len(self.data) - self.lookback - self.horizon + 1

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.lookback]
        # Targets are the next 'horizon' weeks of proportions (all columns except the last one)
        y = self.data[idx + self.lookback : idx + self.lookback + self.horizon, :-1]
        return x, y.reshape(-1)

In [6]:
# --- THE TRAINING FUNCTION ---
def train_freight_model(train_slice, val_slice, config):
    input_dim = train_slice.shape[1]
    num_proportions = input_dim - 1
    output_dim = num_proportions * config['horizon']

    train_ds = FreightDataset(train_slice, config['lookback'], config['horizon'])
    val_ds = FreightDataset(val_slice, config['lookback'], config['horizon'])

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

    model = FreightLSTM(input_dim, config['hidden'], output_dim, config['dropout']).to(device)
    optimizer = optim.Adam(model.parameters(), lr=config['lr'])
    criterion = nn.HuberLoss()

    best_val_loss = float('inf')
    trigger_times = 0
    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(config['epochs']):
        model.train()
        train_loss = 0.0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                val_loss += criterion(model(batch_x), batch_y).item()

        avg_train = train_loss / len(train_loader)
        avg_val = val_loss / len(val_loader)
        history['train_loss'].append(avg_train)
        history['val_loss'].append(avg_val)

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            trigger_times = 0
            torch.save(model.state_dict(), 'champion_production_model.pth')
        else:
            trigger_times += 1
            if trigger_times >= config['patience']:
                print(f"Early stopping at epoch {epoch+1}")
                break

    # Load best weights before returning
    model.load_state_dict(torch.load('champion_production_model.pth'))
    return model, history

# Run Training
train_slice = full_data_matrix_scaled[:split_idx]
val_slice = full_data_matrix_scaled[split_idx - config['lookback'] :]
model, history = train_freight_model(train_slice, val_slice, config)

Early stopping at epoch 25


In [7]:
# --- PRODUCTION FORECAST & EXPORT ---
model.eval()

# 1. Grab the most recent 'lookback' weeks for the final jump
# Use the full history scaling for the final prediction
last_window = full_data_matrix_scaled[-config['lookback']:]
last_window_tensor = torch.tensor(last_window).unsqueeze(0).to(device)

with torch.no_grad():
    future_scaled = model(last_window_tensor).cpu().numpy().reshape(config['horizon'], -1)

# 2. Inverse Transform
# Create dummy column for Total_Volume (last col) to match scaler shape
dummy = np.zeros((config['horizon'], 1))
future_unscaled = scaler.inverse_transform(np.hstack([future_scaled, dummy]))

# 3. Create Wide Forecast DataFrame
last_date = pd.to_datetime(data['Week']).max()
future_dates = pd.date_range(start=last_date + pd.Timedelta(weeks=1),
                             periods=config['horizon'],
                             freq='W')

forecast_wide = pd.DataFrame(future_unscaled[:, :-1], index=future_dates, columns=commodity_names)

# 4. CONVERT TO ENHANCED LONG FORMAT
forecast_long = forecast_wide.reset_index().rename(columns={'index': 'Week'})
forecast_long = forecast_long.melt(id_vars='Week',
                                   var_name='Full_Code',
                                   value_name='Proportion')

# 5. PARSE THE COLUMN NAMES (Originated_BNSF_A -> Type, Company, Code)
# We split by the underscore '_'
# expand=True turns the result into new columns
split_cols = forecast_long['Full_Code'].str.split('_', expand=True)

forecast_long['Type'] = split_cols[0]     # Originated or Received
forecast_long['Company'] = split_cols[1]  # BNSF, CN, CPKC, etc.
forecast_long['Commodity'] = split_cols[2] # A, B, C...

# Clean up: Remove the helper 'Full_Code' and reorder columns for clarity
forecast_long = forecast_long[['Week', 'Type', 'Company', 'Commodity', 'Proportion']]

# 6. Export
forecast_long.to_csv('Next_Quarter_Forecast_.csv', index=False)

print("Success! Enhanced Long-Form Forecast exported.")
print(forecast_long.head())

Success! Enhanced Long-Form Forecast exported.
        Week        Type Company Commodity  Proportion
0 2025-10-05  Originated    BNSF         A    0.010095
1 2025-10-12  Originated    BNSF         A    0.010032
2 2025-10-19  Originated    BNSF         A    0.010689
3 2025-10-26  Originated    BNSF         A    0.010215
4 2025-11-02  Originated    BNSF         A    0.010265


In [8]:
# --- OPTIONAL: SYMPOSIUM VISUALS ---
# (Requires a separate run of actuals vs preds on the val set)
print("Visuals function loaded. You can now use it with your validation predictions.")

Visuals function loaded. You can now use it with your validation predictions.
